# Week 12: Generative AI, Transformers & Diffusion
## Notebook 11: ControlNet Architecture & Zero Convolutions

### 1. Conceptual Intuition & Theoretical Background
Text-to-image diffusion models generate stunning images from prompts, but text prompts cannot precisely convey spatial layout (e.g. human poses, edge sketches, depth boundaries).

**ControlNet (Zhang & Agrawala, 2023):**
Adds spatial conditioning controls to pretrained foundation diffusion models without damaging original capabilities:
1. **Locked Backbone:** Freezes all parameters of the original pretrained U-Net / DiT blocks to preserve foundation generation quality.
2. **Trainable Copy:** Clones the encoder blocks and trains them on task-specific spatial conditions (e.g. Canny edges, OpenPose skeletons).
3. **Zero Convolutions ($\mathcal{Z}$):**
   $1 \times 1$ convolution layers with both weights and biases initialized to **identically zero**:
$$y = \mathcal{F}(x; \Theta) + \mathcal{Z}\left( \mathcal{F}_{\text{copy}}(x + \mathcal{Z}(c); \Theta_{\text{copy}}), \Theta_{z2} \right)$$

**Zero Initialization Benefit:**
At step 0 of training:
$$\mathcal{Z}(c) = 0 \implies y = \mathcal{F}(x; \Theta)$$
The initial output matches the original pretrained model output identically, avoiding destructive random noise injection early in training.

---

### 2. Scratch Python Implementation: Zero Convolution Block

In [ ]:
import numpy as np

class ZeroConvolution2DScratch:
    def __init__(self, channels):
        self.channels = channels
        # Strictly initialize weights and bias to zeros!
        self.weight = np.zeros((channels, channels, 1, 1))
        self.bias = np.zeros(channels)
        
    def forward(self, x):
        """x: shape (batch, channels, H, W)"""
        # 1x1 convolution
        out = np.zeros_like(x)
        # Elementwise linear combination across channels
        for c_out in range(self.channels):
            channel_acc = np.zeros_like(x[:, 0])
            for c_in in range(self.channels):
                channel_acc += x[:, c_in] * self.weight[c_out, c_in, 0, 0]
            out[:, c_out] = channel_acc + self.bias[c_out]
        return out

class ControlNetToyBlock:
    def __init__(self, channels=4):
        self.channels = channels
        # Locked pretrained layer
        np.random.seed(42)
        self.locked_layer = np.random.normal(0, 0.1, (channels, channels))
        
        # Trainable copy layer
        self.trainable_copy = self.locked_layer.copy()
        
        # Zero Convolution layers
        self.zero_conv_in = ZeroConvolution2DScratch(channels)
        self.zero_conv_out = ZeroConvolution2DScratch(channels)
        
    def forward(self, x, condition_hint):
        # 1. Base locked forward output
        base_out = np.dot(x, self.locked_layer)
        
        # 2. Condition path through zero convolutions
        conditioned_input = x + self.zero_conv_in.forward(condition_hint)
        copy_features = np.dot(conditioned_input, self.trainable_copy)
        control_signal = self.zero_conv_out.forward(copy_features)
        
        # 3. Summation
        final_output = base_out + control_signal
        return final_output, base_out

# Test Initial Step Verification
batch_size, channels = 2, 4
x_latent = np.random.randn(batch_size, channels, 8, 8)
edge_condition = np.random.randn(batch_size, channels, 8, 8)

block = ControlNetToyBlock(channels=channels)
# Reshape for 2D forward simulation
x_flat = x_latent.swapaxes(1, 3)
cond_flat = edge_condition.swapaxes(1, 3)

out_combined, out_locked = block.forward(x_flat, cond_flat)
max_discrepancy = np.max(np.abs(out_combined - out_locked))

print("--- ControlNet Zero Convolution Verification ---")
print(f"Max Absolute Difference (Output vs Pretrained Locked): {max_discrepancy:.1e}")
print("At step 0, ControlNet output matches pretrained model identically with zero distortion!")

### 3. Step-by-Step Code Walkthrough
1. **Zero Initialization:** Sets weights and biases of $1 \times 1$ convolutions to zero.
2. **Zero Inflow:** At initialization, $\mathcal{Z}(c) = 0$, ensuring the trainable copy receives unperturbed inputs.
3. **Zero Outflow:** At initialization, the control signal is zero, preserving the original model's generation quality.

---

### 4. Industry Applications & Trade-offs
- **Architectural & Fashion Design:** Rendering realistic photorealistic scenes conditioned on architectural CAD wireframes or pose skeletons.
- **Trade-off:** Cloned encoder weights increase VRAM memory requirements by $\approx 40\%$.

---

### 5. Top Interview Questions & Answers
**Q1: Why are weights in Zero Convolutions initialized to zero instead of standard Xavier/He initialization?**
> **Answer:** If initialized randomly, the newly added control signal would inject random noise into the pretrained feature maps at step 0, destroying the original generation capabilities and requiring prolonged retraining to recover.

**Q2: How does a Zero Convolution learn if its initial weights and outputs are zero?**
> **Answer:** While the forward output is initially zero, the gradient with respect to its weights is non-zero ($\frac{\partial \mathcal{L}}{\partial W} = \frac{\partial \mathcal{L}}{\partial y} \cdot x^T$). In the very first optimizer step, weights update away from zero.